# GPU $V_{xc}$ accuracy versus grid grouping

This notebook uses the carbon-chain/def2-QZVPP stress case from `test_gpu_screened_skala_matches_cpu_on_carbon_chain` to isolate how grouping grid points into GPU4PySCF screening blocks affects Skala's integrated $V_{xc}$.

Grid levels 1 and 2 are evaluated independently, each against a dense CPU calculation on the identical grid and density matrix. Every screened candidate executes GPU4PySCF's real CUDA mask construction and AO evaluation with its installed $10^{-10}$ AO threshold and 4096-point block size.

In [ ]:
from dataclasses import dataclass
from typing import Any
from unittest.mock import patch

import cupy
import numpy as np
import torch
from pyscf import dft, gto
from tests.utils import patch_ao_screening

from skala.functional import load_functional
from skala.functional.base import ExcFunctionalBase
from skala.pyscf import features as features_module
from skala.pyscf.backend import dft_gpu
from skala.pyscf.features import _spatial_grid_permutations
from skala.pyscf.numint import SkalaNumInt

np.set_printoptions(precision=4, suppress=True)


@dataclass(frozen=True)
class Evaluation:
    electron_count: float
    xc_energy: float
    vxc: np.ndarray
    active_ao_counts: np.ndarray


@dataclass(frozen=True)
class GridExperiment:
    level: int
    coords: np.ndarrays
    dense_reference: Evaluation
    groupings: dict[str, list[np.ndarray]]
    rows: list[dict[str, object]]


CARBON_CHAIN = """
C 0.0 0.0 0.0
C 1.4 0.0 0.0
C 2.8 0.0 0.0
C 4.2 0.0 0.0
C 5.6 0.0 0.0
C 7.0 0.0 0.0
"""
GRID_LEVELS = (1, 2)

assert torch.cuda.is_available()
assert dft_gpu is not None
mol = gto.M(atom=CARBON_CHAIN, basis="def2-qzvpp", verbose=0)
dm = dft.RKS(mol).get_init_guess()

cpu_functional = load_functional("skala-1.1", device=torch.device("cpu"))
gpu_functional = load_functional("skala-1.1", device=torch.device("cuda:0"))
assert isinstance(cpu_functional, ExcFunctionalBase)
assert isinstance(gpu_functional, ExcFunctionalBase)
cpu_numint = SkalaNumInt(cpu_functional, device=torch.device("cpu"))
gpu_numint = SkalaNumInt(gpu_functional, device=torch.device("cuda:0"))
GPU_BLOCK_SIZE = int(dft_gpu.numint.MIN_BLK_SIZE)

print(f"Atoms / AOs / shells: {mol.natm} / {mol.nao_nr()} / {mol.nbas}")
print(f"Grid levels: {GRID_LEVELS}")
print(f"GPU4PySCF AO threshold: {dft_gpu.numint.AO_THRESHOLD:.1e}")
print(f"GPU screening block size: {GPU_BLOCK_SIZE}")

## Screening permutations

Each algorithm partitions the complete grid required by that setting: level 1 has 31,080 points in 8 physical GPU groups, while level 2 has 67,248 points in 17 groups. Every group contains at most 4096 points.

The atom-major case preserves PySCF's original grid order and divides the complete sequence into consecutive blocks. The production spatial case recursively partitions the complete coordinate set, choosing split directions from the spatial extent. The mixed cases exchange points between complete spatial groups while preserving every group size and leaving the final partial group intact. In every case, each grid point appears exactly once.

Even spatially grouped 4096-point blocks can overlap many functions in a diffuse def2-QZVPP basis. `Active AO fraction` reports the grid-point-weighted fraction of AOs retained by the actual masks. `DM matmul proxy` weights the squared fraction, matching the leading $n_{\mathrm{active}}^2 n_{\mathrm{grid}}$ scaling of Skala's density-feature matrix multiplications; neither column is a measured runtime.

In [ ]:
def validate_partition(groups: list[np.ndarray], point_count: int) -> None:
    if not groups or any(group.size == 0 for group in groups):
        raise ValueError("Groups must be non-empty.")
    flattened = np.concatenate(groups)
    if flattened.size != point_count or not np.array_equal(
        np.sort(flattened), np.arange(point_count, dtype=np.int64)
    ):
        raise ValueError("Groups must partition every grid point exactly once.")


def groups_from_permutation(
    permutation: np.ndarray, block_size: int
) -> list[np.ndarray]:
    groups = [
        permutation[start : start + block_size].copy()
        for start in range(0, permutation.size, block_size)
    ]
    validate_partition(groups, permutation.size)
    return groups


def mix_spatial_groups(
    groups: list[np.ndarray], mixing_fraction: float, point_count: int
) -> list[np.ndarray]:
    if not 0 <= mixing_fraction <= 1:
        raise ValueError("mixing_fraction must be between zero and one.")

    complete = [group for group in groups if group.size == GPU_BLOCK_SIZE]
    remainders = [group.copy() for group in groups if group.size != GPU_BLOCK_SIZE]
    if mixing_fraction == 0 or len(complete) < 2:
        return [group.copy() for group in groups]

    source = np.stack(complete)
    mixed = source.copy()
    mixed_columns = round(mixing_fraction * GPU_BLOCK_SIZE)
    columns = np.floor(
        np.arange(mixed_columns) * GPU_BLOCK_SIZE / mixed_columns
    ).astype(np.int64)
    for column_index, column in enumerate(columns):
        shift = 1 + column_index % (source.shape[0] - 1)
        mixed[:, column] = np.roll(source[:, column], shift)

    result = [row.copy() for row in mixed] + remainders
    validate_partition(result, point_count)
    return result


def build_matching_grids(level: int) -> tuple[Any, Any, np.ndarray]:
    cpu_grids = dft.Grids(mol)
    cpu_grids.level = level
    cpu_grids.alignment = 1
    cpu_grids.build(sort_grids=False)
    assert cpu_grids.coords is not None and cpu_grids.weights is not None

    gpu_grids = dft_gpu.Grids(mol)
    gpu_grids.level = level
    gpu_grids.alignment = 1
    gpu_grids.build(sort_grids=False)

    coords = np.asarray(cpu_grids.coords)
    np.testing.assert_allclose(
        coords, cupy.asnumpy(gpu_grids.coords), rtol=0.0, atol=0.0
    )
    np.testing.assert_allclose(
        cpu_grids.weights,
        cupy.asnumpy(gpu_grids.weights),
        rtol=1e-12,
        atol=1e-12,
    )
    return cpu_grids, gpu_grids, coords


def dense_cpu_reference(cpu_grids: Any) -> Evaluation:
    with patch_ao_screening(False):
        electron_count, xc_energy, vxc = cpu_numint.nr_rks(mol, cpu_grids, None, dm)
    return Evaluation(
        electron_count=float(electron_count),
        xc_energy=float(xc_energy),
        vxc=np.asarray(vxc),
        active_ao_counts=np.asarray([mol.nao_nr()], dtype=np.int64),
    )


def fresh_gpu_grids(template: Any) -> Any:
    case_grids = dft_gpu.Grids(mol)
    case_grids.level = template.level
    case_grids.alignment = template.alignment
    case_grids.coords = template.coords
    case_grids.weights = template.weights
    case_grids._non0ao_idx = None
    return case_grids


def evaluate_gpu_permutation(
    permutation: np.ndarray, gpu_grid_template: Any, point_count: int
) -> Evaluation:
    inverse = np.empty_like(permutation)
    inverse[permutation] = np.arange(point_count, dtype=np.int64)
    case_grids = fresh_gpu_grids(gpu_grid_template)
    with (
        patch.object(
            features_module,
            "_spatial_grid_permutations",
            return_value=(permutation, inverse),
        ),
        patch_ao_screening(True),
    ):
        electron_count, xc_energy, vxc = gpu_numint.nr_rks(
            mol, case_grids, None, cupy.asarray(dm)
        )

    prepared_grids, cached_forward, _ = features_module._prepare_spatially_sorted_grids(
        mol, case_grids, GPU_BLOCK_SIZE, gpu=True
    )
    assert np.array_equal(cached_forward, permutation)
    active_ao_counts = np.asarray(
        [entry[1].size for entry in prepared_grids.get_non0ao_idx()],
        dtype=np.int64,
    )
    return Evaluation(
        electron_count=float(electron_count),
        xc_energy=float(xc_energy),
        vxc=cupy.asnumpy(vxc),
        active_ao_counts=active_ao_counts,
    )


def vxc_errors(
    candidate: np.ndarray, dense_reference: Evaluation
) -> tuple[float, float]:
    difference = candidate - dense_reference.vxc
    return (
        float(np.max(np.abs(difference))),
        float(np.linalg.norm(difference) / np.linalg.norm(dense_reference.vxc)),
    )


def normalized_within_group_radius(
    groups: list[np.ndarray], coords: np.ndarray
) -> float:
    global_center = coords.mean(axis=0)
    global_rms = np.sqrt(np.mean(np.sum(np.square(coords - global_center), axis=1)))
    within_sum = 0.0
    for group in groups:
        group_coords = coords[group]
        center = group_coords.mean(axis=0)
        within_sum += float(np.sum(np.square(group_coords - center)))
    return float(np.sqrt(within_sum / coords.shape[0]) / global_rms)


def mean_maximum_bbox_iou(groups: list[np.ndarray], coords: np.ndarray) -> float:
    if len(groups) == 1:
        return 0.0
    minimums = np.asarray([coords[group].min(axis=0) for group in groups])
    maximums = np.asarray([coords[group].max(axis=0) for group in groups])
    volumes = np.prod(np.maximum(maximums - minimums, 0.0), axis=1)
    maximum_ious = []
    for index in range(len(groups)):
        intersection_extent = np.maximum(
            np.minimum(maximums[index], maximums)
            - np.maximum(minimums[index], minimums),
            0.0,
        )
        intersection = np.prod(intersection_extent, axis=1)
        union = volumes[index] + volumes - intersection
        iou = np.divide(
            intersection,
            union,
            out=np.zeros_like(intersection),
            where=union > 0,
        )
        iou[index] = 0.0
        maximum_ious.append(float(iou.max()))
    return float(np.mean(maximum_ious))


def summarize_case(
    level: int,
    name: str,
    groups: list[np.ndarray],
    evaluation: Evaluation,
    coords: np.ndarray,
    dense_reference: Evaluation,
) -> dict[str, object]:
    point_count = coords.shape[0]
    validate_partition(groups, point_count)
    sizes = np.asarray([group.size for group in groups], dtype=np.int64)
    active_aos = evaluation.active_ao_counts
    assert sizes.size == active_aos.size
    maximum_error, relative_error = vxc_errors(evaluation.vxc, dense_reference)
    return {
        "level": level,
        "grid_points": point_count,
        "case": name,
        "groups": len(groups),
        "occupancy": f"{sizes.min()}/{np.median(sizes):.0f}/{sizes.max()}",
        "active_aos": f"{active_aos.min()}/{np.median(active_aos):.0f}/{active_aos.max()}",
        "radius": normalized_within_group_radius(groups, coords),
        "bbox_iou": mean_maximum_bbox_iou(groups, coords),
        "active_ao_fraction": float(
            np.sum(sizes * active_aos) / (point_count * mol.nao_nr())
        ),
        "dm_matmul_proxy": float(
            np.sum(sizes * np.square(active_aos)) / (point_count * mol.nao_nr() ** 2)
        ),
        "max_vxc_error": maximum_error,
        "relative_vxc_error": relative_error,
        "electron_error": abs(
            evaluation.electron_count - dense_reference.electron_count
        ),
        "energy_error": abs(evaluation.xc_energy - dense_reference.xc_energy),
    }


def build_groupings(coords: np.ndarray) -> dict[str, list[np.ndarray]]:
    point_count = coords.shape[0]
    atom_major = groups_from_permutation(
        np.arange(point_count, dtype=np.int64), GPU_BLOCK_SIZE
    )
    spatial_forward, _ = _spatial_grid_permutations(coords, GPU_BLOCK_SIZE)
    spatial = groups_from_permutation(spatial_forward, GPU_BLOCK_SIZE)
    return {
        "GPU4PySCF atom-major blocks": atom_major,
        "GPU4PySCF spatial blocks": spatial,
        "GPU4PySCF spatial, mix 0.500": mix_spatial_groups(spatial, 0.5, point_count),
        "GPU4PySCF spatial, mix 1.000": mix_spatial_groups(spatial, 1.0, point_count),
    }


def run_grid_level(level: int) -> GridExperiment:
    cpu_grids, gpu_grid_template, coords = build_matching_grids(level)
    point_count = coords.shape[0]
    dense_reference = dense_cpu_reference(cpu_grids)
    groupings = build_groupings(coords)
    all_points = [np.arange(point_count, dtype=np.int64)]
    case_data = [("Dense CPU reference", all_points, dense_reference)]

    print(f"Level {level}: {point_count:,} points")
    for name, groups in groupings.items():
        print(f"  Evaluating {name}...")
        evaluation = evaluate_gpu_permutation(
            np.concatenate(groups), gpu_grid_template, point_count
        )
        assert np.isfinite(evaluation.electron_count)
        assert np.isfinite(evaluation.xc_energy)
        assert np.isfinite(evaluation.vxc).all()
        assert np.allclose(evaluation.vxc, evaluation.vxc.T, rtol=1e-10, atol=1e-11)
        case_data.append((name, groups, evaluation))

    rows = [
        summarize_case(level, name, groups, evaluation, coords, dense_reference)
        for name, groups, evaluation in case_data
    ]
    atom_major_error = rows[1]["max_vxc_error"]
    spatial_error = rows[2]["max_vxc_error"]
    assert isinstance(atom_major_error, float)
    assert isinstance(spatial_error, float)
    print(
        "  Spatial grouping changes max |dVxc| by "
        f"{atom_major_error / spatial_error:.2f}x."
    )
    return GridExperiment(level, coords, dense_reference, groupings, rows)


def render_results(rows: list[dict[str, object]]) -> str:
    columns = (
        ("level", "Grid level"),
        ("grid_points", "Grid points"),
        ("case", "Case"),
        ("groups", "Groups"),
        ("occupancy", "Points min/med/max"),
        ("active_aos", "Active AOs min/med/max"),
        ("radius", "RMS radius"),
        ("bbox_iou", "BBox IoU"),
        ("active_ao_fraction", "Active AO fraction"),
        ("dm_matmul_proxy", "DM matmul proxy"),
        ("max_vxc_error", "max |dVxc|"),
        ("relative_vxc_error", "rel. Frobenius"),
        ("electron_error", "|dN|"),
        ("energy_error", "|dExc|"),
    )
    parts = [
        '<table style="border-collapse:collapse;font-variant-numeric:tabular-nums">',
        "<thead><tr>",
    ]
    parts.extend(
        f'<th style="padding:4px 8px;border-bottom:1px solid #888">{label}</th>'
        for _, label in columns
    )
    parts.append("</tr></thead><tbody>")
    for row in rows:
        parts.append("<tr>")
        for key, _ in columns:
            value = row[key]
            text = f"{value:.3e}" if isinstance(value, float) else str(value)
            parts.append(
                f'<td style="padding:3px 8px;border-bottom:1px solid #ddd">{text}</td>'
            )
        parts.append("</tr>")
    parts.append("</tbody></table>")
    return "".join(parts)


class HTMLTable(str):
    def _repr_html_(self) -> str:
        return str(self)

## Results

Each grid level has its own dense CPU reference and independently constructed grouping permutations. All screened rows use actual GPU4PySCF masks. Lower active-AO metrics mean more aggressive screening; lower error means closer agreement with that level's dense reference.

In [ ]:
experiments = [run_grid_level(level) for level in GRID_LEVELS]
results = [row for experiment in experiments for row in experiment.rows]

HTMLTable(render_results(results))

## Fixed grid-group slices

For each grid level, the figure shows three fixed slabs centered at $z=-1$, $0$, and $+1$ bohr relative to the molecular $x$-$y$ plane. Each slab includes points satisfying $|z-z_0|\leq 0.25$ bohr. The outermost 1% of each grid, ranked by three-dimensional distance to the nearest carbon nucleus, is omitted to keep the molecular region legible.

The selected points are accumulated in shared $x$-$y$ bins. Each occupied bin takes the color of its most frequent screening group; the color is blended toward white according to that group's fraction of points in the bin. Pure color means complete local agreement, while a pale bin contains a stronger mixture of groups. Black crosses mark the carbon nuclei.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap


def group_labels(groups: list[np.ndarray], point_count: int) -> np.ndarray:
    labels = np.empty(point_count, dtype=np.int64)
    for group_index, group in enumerate(groups):
        labels[group] = group_index
    return labels


def dominant_group_image(
    coords: np.ndarray,
    labels: np.ndarray,
    selected: np.ndarray,
    x_edges: np.ndarray,
    y_edges: np.ndarray,
    group_colors: np.ndarray,
) -> np.ndarray:
    x_bin_count = x_edges.size - 1
    y_bin_count = y_edges.size - 1
    selected_coords = coords[selected]
    x_bins = np.searchsorted(x_edges, selected_coords[:, 0], side="right") - 1
    y_bins = np.searchsorted(y_edges, selected_coords[:, 1], side="right") - 1
    x_bins = np.clip(x_bins, 0, x_bin_count - 1)
    y_bins = np.clip(y_bins, 0, y_bin_count - 1)
    flat_bins = y_bins * x_bin_count + x_bins
    combined = labels[selected] * (x_bin_count * y_bin_count) + flat_bins
    counts = np.bincount(
        combined,
        minlength=group_colors.shape[0] * x_bin_count * y_bin_count,
    ).reshape(group_colors.shape[0], y_bin_count, x_bin_count)
    assert int(counts.sum()) == int(selected.sum())

    totals = counts.sum(axis=0)
    dominant_groups = counts.argmax(axis=0)
    dominant_counts = counts.max(axis=0)
    populated = totals > 0
    dominant_fraction = np.divide(
        dominant_counts,
        totals,
        out=np.zeros_like(dominant_counts, dtype=float),
        where=populated,
    )

    image = np.ones((*totals.shape, 4), dtype=float)
    dominant_colors = group_colors[dominant_groups]
    image[populated, :3] = 1.0 - dominant_fraction[populated, None] * (
        1.0 - dominant_colors[populated]
    )
    return image


def rgb_to_lab(rgb: np.ndarray) -> np.ndarray:
    linear = np.where(
        rgb <= 0.04045,
        rgb / 12.92,
        ((rgb + 0.055) / 1.055) ** 2.4,
    )
    transform = np.asarray(
        [
            [0.4124564, 0.3575761, 0.1804375],
            [0.2126729, 0.7151522, 0.0721750],
            [0.0193339, 0.1191920, 0.9503041],
        ]
    )
    xyz = linear @ transform.T
    xyz /= np.asarray([0.95047, 1.0, 1.08883])
    delta = 6 / 29
    transformed = np.where(
        xyz > delta**3,
        np.cbrt(xyz),
        xyz / (3 * delta**2) + 4 / 29,
    )
    return np.column_stack(
        (
            116 * transformed[:, 1] - 16,
            500 * (transformed[:, 0] - transformed[:, 1]),
            200 * (transformed[:, 1] - transformed[:, 2]),
        )
    )


def distinct_group_colors(count: int) -> np.ndarray:
    levels = np.linspace(0.0, 1.0, 11)
    candidates = np.stack(
        np.meshgrid(levels, levels, levels, indexing="ij"), axis=-1
    ).reshape(-1, 3)
    candidate_lab = rgb_to_lab(candidates)
    chroma = np.linalg.norm(candidate_lab[:, 1:], axis=1)
    keep = (candidate_lab[:, 0] >= 35) & (candidate_lab[:, 0] <= 75) & (chroma >= 35)
    candidates = candidates[keep]
    candidate_lab = candidate_lab[keep]

    seed = np.argmin(np.linalg.norm(candidates - np.asarray([0.0, 0.3, 0.8]), axis=1))
    selected = [int(seed)]
    minimum_distance = np.linalg.norm(candidate_lab - candidate_lab[seed], axis=1)
    for _ in range(1, count):
        index = int(np.argmax(minimum_distance))
        selected.append(index)
        distance = np.linalg.norm(candidate_lab - candidate_lab[index], axis=1)
        minimum_distance = np.minimum(minimum_distance, distance)
    return candidates[selected]


atom_coords = mol.atom_coords()
slice_centers = (-1.0, 0.0, 1.0)
slice_half_width = 0.25
retained_by_level = {}
for experiment in experiments:
    nearest_atom_distance = np.linalg.norm(
        experiment.coords[:, None, :] - atom_coords[None, :, :], axis=2
    ).min(axis=1)
    removed_count = round(0.01 * experiment.coords.shape[0])
    retained = np.ones(experiment.coords.shape[0], dtype=bool)
    outside_order = np.argsort(nearest_atom_distance, kind="stable")
    retained[outside_order[-removed_count:]] = False
    assert retained.sum() == experiment.coords.shape[0] - removed_count
    retained_by_level[experiment.level] = retained

trimmed_xy = np.concatenate(
    [
        experiment.coords[retained_by_level[experiment.level], :2]
        for experiment in experiments
    ]
)
x_min, y_min = trimmed_xy.min(axis=0)
x_max, y_max = trimmed_xy.max(axis=0)
x_bin_count = 120
bin_width = (x_max - x_min) / x_bin_count
y_bin_count = max(1, int(np.ceil((y_max - y_min) / bin_width)))
y_center = 0.5 * (y_min + y_max)
x_edges = np.linspace(x_min, x_max, x_bin_count + 1)
y_edges = np.linspace(
    y_center - 0.5 * y_bin_count * bin_width,
    y_center + 0.5 * y_bin_count * bin_width,
    y_bin_count + 1,
)

for experiment in experiments:
    coords = experiment.coords
    retained = retained_by_level[experiment.level]
    group_count = max(len(groups) for groups in experiment.groupings.values())
    group_colors = distinct_group_colors(group_count)
    palette = ListedColormap(group_colors)
    norm = BoundaryNorm(np.arange(group_count + 1) - 0.5, palette.N)
    labels_by_name = {
        name: group_labels(groups, coords.shape[0])
        for name, groups in experiment.groupings.items()
    }

    figure, axes = plt.subplots(
        len(labels_by_name),
        len(slice_centers),
        figsize=(15, 13),
        sharex=True,
        sharey=True,
        constrained_layout=True,
        squeeze=False,
    )
    for row_index, (name, labels) in enumerate(labels_by_name.items()):
        for column_index, height in enumerate(slice_centers):
            axis = axes[row_index, column_index]
            selected = retained & (np.abs(coords[:, 2] - height) <= slice_half_width)
            image = dominant_group_image(
                coords,
                labels,
                selected,
                x_edges,
                y_edges,
                group_colors,
            )
            axis.imshow(
                image,
                origin="lower",
                extent=(x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]),
                interpolation="nearest",
                aspect="equal",
            )
            axis.scatter(
                atom_coords[:, 0],
                atom_coords[:, 1],
                marker="x",
                c="black",
                s=24,
                linewidths=1.0,
                zorder=3,
            )
            axis.text(
                0.98,
                0.96,
                f"{selected.sum():,} points",
                ha="right",
                va="top",
                transform=axis.transAxes,
                fontsize=8,
            )
            if row_index == 0:
                axis.set_title(
                    f"z = {height:+.1f} +/- {slice_half_width:.2f} bohr",
                    fontsize=10,
                )
            if column_index == 0:
                axis.set_ylabel(f"{name}\ny (bohr)", fontsize=9)
            if row_index == len(labels_by_name) - 1:
                axis.set_xlabel("x (bohr)")

    colorbar = figure.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=palette),
        ax=axes,
        ticks=np.arange(group_count),
        shrink=0.82,
        pad=0.02,
    )
    colorbar.ax.set_yticklabels(np.arange(1, group_count + 1))
    colorbar.set_label("Dominant screening group")
    figure.suptitle(
        f"Level {experiment.level}: dominant GPU screening groups "
        f"({coords.shape[0]:,} grid points)"
    )
    plt.show()